In [16]:
import pandas as pd
import os
import glob

# Define paths consistent with the R script
target = 'tceq'
input_file = f"D:/tierra/datasets/Mexico_wosis_cleaned_{target}.csv"
harmonized_dir =f"D:/tierra/outputs/harmonized/{target}"
output_file = f"D:/tierra/outputs/harmonized/Mexico_standardized_{target}.csv"

# Define soil features to process (same as in R script)
soil_features = ["clay", "phaq", "sand", "silt", target]

print("Loading original soil dataset...")
# Read the original dataset
original_data = pd.read_csv(input_file)
print(f"Original dataset loaded with {len(original_data)} records")

# Create a dictionary to store harmonized data for each feature
harmonized_features = {}

# Load harmonized data for each feature
for feature in soil_features:
    feature_file = os.path.join(harmonized_dir, f"{feature}_harmonized.csv")
    if os.path.exists(feature_file):
        print(f"Loading harmonized data for {feature}...")
        harmonized_features[feature] = pd.read_csv(feature_file)
    else:
        print(f"Harmonized data file for {feature} not found. Skipping.")

harmonized_features

Loading original soil dataset...
Original dataset loaded with 2621 records
Loading harmonized data for clay...
Loading harmonized data for phaq...
Loading harmonized data for sand...
Loading harmonized data for silt...
Loading harmonized data for tceq...


{'clay':            id     0-5 cm    5-15 cm     15-30 cm  soil depth
 0     1149108   4.000000   4.000000     4.000000          21
 1     1149115  12.000000  12.000000    12.000000          22
 2     1149116  10.000000  10.000000 -9999.000000          15
 3     1149117   2.000000   2.000000     2.000000          20
 4     1149118  20.000000  20.000000    20.000000          22
 ...       ...        ...        ...          ...         ...
 2145  1671286  16.000000  16.000000    16.000000          24
 2146  1671289  14.000000  14.000000    14.000000          17
 2147  1671313  18.000000  18.000000    18.000000          26
 2148  1671318  21.096631  22.526152    25.805287          27
 2149  1671323   2.000000   2.000000     2.000000          30
 
 [2150 rows x 5 columns],
 'phaq':            id    0-5 cm   5-15 cm     15-30 cm  soil depth
 0     1149108  5.500000  5.500000     5.500000          21
 1     1149115  5.900000  5.900000     5.900000          22
 2     1149116  5.900000  5.9000

In [17]:
# Get unique profile IDs
profile_ids = original_data['profile_id'].unique()

# Initialize lists to store data
rows = []

depth_categories = ['0-5', '5-15', '15-30']

# Process each profile
for profile_id in profile_ids:
    # For each depth interval
    for depth_cat in depth_categories:
        row = {'profile_id': profile_id, 'depth_category': depth_cat}
        
        # Add basic location info from original data
        profile_data = original_data[original_data['profile_id'] == profile_id].iloc[0]
        row.update({
            # 'upper_depth': profile_data['upper_depth'],
            # 'lower_depth': profile_data['lower_depth'],
            'date': profile_data['date'],
            'longitude': profile_data['longitude'],
            'latitude': profile_data['latitude']
        })
        
        # Add each feature's value for this depth
        for feature in soil_features:
            feature_df = harmonized_features[feature]
            if profile_id in feature_df['id'].values:
                depth_map = {
                    '0-5': '0-5 cm',
                    '5-15': '5-15 cm',
                    '15-30': '15-30 cm'
                }
                value = feature_df[feature_df['id'] == profile_id][depth_map[depth_cat]].iloc[0]
                row[feature] = value
            else:
                row[feature] = None
        
        rows.append(row)

# Create final dataframe
final_df = pd.DataFrame(rows)


In [18]:
final_df.head()

# print row where profile_id is 1
# final_df[final_df['profile_id'] == 1147760].head()

,profile_id,depth_category,date,longitude,latitude,clay,phaq,sand,silt,tceq
0,1358127,0-5,1969-8-11,-100.462493,21.496251,40.000000,6.300000,44.000000,16.000000,0.0
1,1358127,5-15,1969-8-11,-100.462493,21.496251,40.000000,6.300000,44.000000,16.000000,0.0
2,1358127,15-30,1969-8-11,-100.462493,21.496251,-9999.000000,-9999.000000,-9999.000000,-9999.000000,-9999.0
3,1601435,0-5,1969-8-15,-101.695887,21.741673,21.448630,6.058647,48.275685,30.275685,0.0
4,1601435,5-15,1969-8-15,-101.695887,21.741673,23.247432,6.193557,47.376284,29.376284,0.0


In [19]:
# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Save the final dataframe
final_df.to_csv(output_file, index=False)
print(f"Merged data saved to: {output_file}")

Merged data saved to: D:/tierra/outputs/harmonized/Mexico_standardized_tceq.csv
